# 03 · Temporal Analysis

**Amaç:** Fraud oranındaki temporal hareketi belgelemek; time-based split kararını gerekçelendirmek.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated


In [ ]:
df, _ = load_validated()
df['_q'] = df['TransactionDate'].dt.to_period('Q')
df['_m'] = df['TransactionDate'].dt.to_period('M')
df['_hour'] = df['TransactionDate'].dt.hour
df['_dow'] = df['TransactionDate'].dt.dayofweek

## Çeyreklik fraud oranı

In [ ]:
q = df.groupby('_q').agg(n=('IsFraudTransaction','size'), fraud=('IsFraudTransaction','sum'))
q['rate_pct'] = (q['fraud']/q['n']*100).round(3)
print(q.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
q['n'].plot.bar(ax=ax1, title='Quarterly volume')
q['rate_pct'].plot.bar(ax=ax2, title='Quarterly fraud rate (%)', color='crimson')
plt.tight_layout()

## Aylık trend & rolling-30d fraud rate

In [ ]:
m = df.groupby('_m').agg(n=('IsFraudTransaction','size'), fraud=('IsFraudTransaction','sum'))
m['rate_pct'] = (m['fraud']/m['n']*100)
fig, ax = plt.subplots(figsize=(12,4))
m['rate_pct'].plot(ax=ax, marker='o')
ax.set_title('Monthly fraud rate (%) — drift büyük')
ax.set_ylabel('%')

## Saat ve gün-haftası — düz mü?

In [ ]:
h = df.groupby('_hour')['IsFraudTransaction'].mean()*100
d = df.groupby('_dow')['IsFraudTransaction'].mean()*100
fig, axes = plt.subplots(1,2, figsize=(14,4))
h.plot.bar(ax=axes[0], title='Fraud rate by hour (%)')
d.plot.bar(ax=axes[1], title='Fraud rate by day-of-week (%)')
for ax in axes: ax.set_ylim(0, max(h.max(), d.max())*1.5)

## Sonuç
- Çeyreklik fraud oranı 0.00% → 1.26% → 0.13% arasında oynuyor. Bu büyük drift split stratejisini belirler.
- Saat ve gün-haftası tamamen düz — sentetik veri sinyalı. Bu kolonların production'da değer üretmesi beklenmemeli.